In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "hls_fde_dev", "UC Catalog")
dbutils.widgets.text("schema", "dev_matthew_giglia_semantic_ground_truth", "UC Schema")
dbutils.widgets.text("warehouse_id", "0ea84986a23a47c3", "SQL Warehouse ID")
dbutils.widgets.text("space_title", "Ground Truth Operations", "Genie Space Title")
dbutils.widgets.text("dashboard_title", "Semantic Ground Truth — Operations", "Operations Dashboard Title")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
warehouse_id = dbutils.widgets.get("warehouse_id")
space_title = dbutils.widgets.get("space_title")
dashboard_title = dbutils.widgets.get("dashboard_title")

print(f"Catalog:      {catalog}")
print(f"Schema:       {schema}")
print(f"Warehouse ID: {warehouse_id}")
print(f"Space Title:  {space_title}")
print(f"Dashboard:    {dashboard_title}")

In [0]:
import json

METRIC_VIEWS = [
    "mv_review_activity",
    "mv_coverage_metrics",
    "mv_user_leaderboard",
    "mv_feedback_pipeline",
    "mv_campaign_progress",
    "mv_otel_error_rate",
    "mv_otel_request_latency",
]

results = []


def _find_first_measure(fqn: str) -> str | None:
    """Return the name of the first measure in a metric view."""
    try:
        raw = spark.sql(f"DESCRIBE TABLE EXTENDED {fqn} AS JSON").collect()[0][0]
        spec = json.loads(raw)
        # Measures appear as columns with is_measure: true
        for col in spec.get("columns", []):
            if col.get("is_measure"):
                return col.get("name")
    except Exception:
        pass
    return None


for mv_name in METRIC_VIEWS:
    fqn = f"`{catalog}`.`{schema}`.`{mv_name}`"
    row = {"check": f"metric_view:{mv_name}", "passed": False, "detail": ""}
    try:
        spark.sql(f"DESCRIBE TABLE {fqn}").collect()
        measure = _find_first_measure(fqn)
        if measure:
            spark.sql(f"SELECT MEASURE(`{measure}`) FROM {fqn}").collect()
            row["passed"] = True
            row["detail"] = f"MEASURE(`{measure}`) succeeded"
        else:
            row["detail"] = "View exists but no measure discovered via DESCRIBE AS JSON"
    except Exception as e:
        row["detail"] = str(e)[:300]
    results.append(row)

print(f"Metric views: {sum(1 for r in results if r['passed'])}/{len(METRIC_VIEWS)} passed")

In [0]:
bronze_fqn = f"`{catalog}`.`{schema}`.bronze_lakebase_cdf"

for source_table in ("votes", "assets"):
    row = {"check": f"bronze_cdf:{source_table}", "passed": False, "detail": ""}
    try:
        cnt = spark.sql(
            f"SELECT COUNT(*) AS cnt FROM {bronze_fqn} WHERE _source_table = '{source_table}'"
        ).collect()[0]["cnt"]
        row["passed"] = cnt > 0
        row["detail"] = f"{cnt} rows"
    except Exception as e:
        row["detail"] = str(e)[:300]
    results.append(row)

print(f"Bronze CDF checks done")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
space_id = None

row = {"check": "genie_space:exists_and_warehouse", "passed": False, "detail": ""}
try:
    resp = w.api_client.do("GET", "/api/2.0/genie/spaces")
    spaces = resp.get("spaces", [])
    matching = [s for s in spaces if s.get("title", "").endswith(space_title)]
    if not matching:
        row["detail"] = f"No space titled '{space_title}' found among {len(spaces)} spaces"
    elif len(matching) > 1:
        row["detail"] = f"Multiple spaces ({len(matching)}) titled '{space_title}'"
    else:
        space = matching[0]
        space_id = space.get("space_id") or space.get("id")
        sw = space.get("warehouse_id", "")
        if sw == warehouse_id:
            row["passed"] = True
            row["detail"] = f"id={space_id}, warehouse OK"
        else:
            row["detail"] = f"id={space_id}, warehouse mismatch: {sw} != {warehouse_id}"
except Exception as e:
    row["detail"] = str(e)[:300]
results.append(row)

print(f"Genie space: {'PASS' if row['passed'] else 'FAIL'} — {row['detail']}")

In [0]:
row = {"check": "genie_conversation:votes_question", "passed": False, "detail": ""}

if space_id is None:
    row["detail"] = "Skipped — Genie space not found"
else:
    try:
        conv = w.genie.start_conversation_and_wait(
            space_id=space_id,
            content="How many votes were cast in total?",
        )
        conversation_id = conv.conversation_id
        message_id = conv.id

        msg = w.genie.get_message(
            space_id=space_id,
            conversation_id=conversation_id,
            message_id=message_id,
        )

        for att in msg.attachments or []:
            if att.query and att.query.query:
                qr = w.genie.get_message_attachment_query_result(
                    space_id=space_id,
                    conversation_id=conversation_id,
                    message_id=message_id,
                    attachment_id=att.attachment_id,
                )
                # Check for SUCCEEDED status — handle both object and dict forms
                status_str = ""
                if hasattr(qr, "statement_response"):
                    sr = qr.statement_response
                    if hasattr(sr, "status"):
                        status_str = str(getattr(sr.status, "state", sr.status))
                elif hasattr(qr, "status"):
                    status_str = str(qr.status)
                if "SUCCEEDED" in status_str.upper():
                    row["passed"] = True
                    row["detail"] = "SQL attachment query SUCCEEDED"
                    break

        if not row["passed"] and not row["detail"]:
            row["detail"] = "No successful SQL query attachment in response"
    except Exception as e:
        row["detail"] = str(e)[:300]

results.append(row)
print(f"Genie conversation: {'PASS' if row['passed'] else 'FAIL'} — {row['detail']}")

In [0]:
# Operations AI/BI dashboard (resources/ops_dashboard.dashboard.yml, session 29):
# it exists, is published, and every dataset executes on the warehouse with its default parameters.
import re
import time
from datetime import date, timedelta

from databricks.sdk.service.sql import StatementParameterListItem

_SQL_TERMINAL = ("SUCCEEDED", "FAILED", "CANCELED", "CLOSED")
_PARAM_TYPES = {"INTEGER": "INT", "DATETIME": "TIMESTAMP"}  # Lakeview dataType -> SQL type
_UNIT_DAYS = {"d": 1, "w": 7, "M": 30, "y": 365}


def _resolve_date(expr: str) -> str:
    """Lakeview relative date (now, now-30d/d, now-1w/w) -> ISO date; absolute dates pass through."""
    m = re.fullmatch(r"now(?:([+-]\d+)([dwMy]))?(?:/[dwMy])?", (expr or "").strip())
    if not m:
        return str(expr)[:10]
    d = date.today()
    if m.group(1):
        d += timedelta(days=int(m.group(1)) * _UNIT_DAYS[m.group(2)])
    return d.isoformat()


def _bind_params(query: str, ds: dict):
    """Inline RANGE parameter defaults (:p.min / :p.max / :p IS NULL); single values become statement parameters."""
    items, unsupported = [], []
    for p in ds.get("parameters") or []:
        kw = p["keyword"]
        sel = p.get("defaultSelection") or {}
        if p.get("complexType") == "RANGE" and "range" in sel:
            rng = sel["range"]
            lo = _resolve_date((rng.get("min") or {}).get("value", ""))
            hi = _resolve_date((rng.get("max") or {}).get("value", ""))
            query = re.sub(rf":{kw}\.min\b", f"DATE'{lo}'", query)
            query = re.sub(rf":{kw}\.max\b", f"DATE'{hi}'", query)
            query = re.sub(rf":{kw}\s+IS\s+NULL", "FALSE", query, flags=re.IGNORECASE)
            continue
        vals = ((sel.get("values") or {}).get("values")) or []
        if len(vals) == 1 and "value" in vals[0]:
            t = (p.get("dataType") or "STRING").upper()
            items.append(StatementParameterListItem(name=kw, value=str(vals[0]["value"]), type=_PARAM_TYPES.get(t, t)))
        else:
            unsupported.append(kw)
    return query, items, unsupported


def _run_statement(stmt: str, params, timeout_s: int = 180):
    r = w.statement_execution.execute_statement(
        statement=stmt, warehouse_id=warehouse_id, catalog=catalog, schema=schema,
        parameters=params or None, wait_timeout="30s")
    deadline = time.time() + timeout_s
    state = r.status.state.value if r.status and r.status.state else ""
    while state not in _SQL_TERMINAL:
        if time.time() > deadline:
            w.statement_execution.cancel_execution(r.statement_id)
            return "TIMEOUT", f"no result after {timeout_s}s"
        time.sleep(3)
        r = w.statement_execution.get_statement(r.statement_id)
        state = r.status.state.value if r.status and r.status.state else ""
    err = r.status.error.message if r.status and r.status.error else ""
    return state, err


dashboard_id = None
row = {"check": "ops_dashboard:exists_and_published", "passed": False, "detail": ""}
try:
    matching = [d for d in w.lakeview.list() if (d.display_name or "").endswith(dashboard_title)]
    if len(matching) != 1:
        row["detail"] = f"{len(matching)} dashboards titled '{dashboard_title}' (expected 1)"
    else:
        dashboard_id = matching[0].dashboard_id
        pub = w.lakeview.get_published(dashboard_id)
        row["passed"] = True
        row["detail"] = f"id={dashboard_id}, published {pub.revision_create_time}, warehouse={pub.warehouse_id}"
except Exception as e:
    row["detail"] = f"id={dashboard_id}: {str(e)[:250]}"
results.append(row)
print(f"Ops dashboard: {'PASS' if row['passed'] else 'FAIL'} — {row['detail']}")

if dashboard_id and row["passed"]:
    datasets = json.loads(w.lakeview.get(dashboard_id).serialized_dashboard or "{}").get("datasets") or []
    if not datasets:
        results.append({"check": "ops_dashboard:datasets", "passed": False, "detail": "dashboard has no datasets"})
    for ds in datasets:
        drow = {"check": f"ops_dashboard:dataset:{ds.get('name')}", "passed": False, "detail": ""}
        try:
            query = "".join(ds.get("queryLines") or []).strip().rstrip(";").strip()
            query, params, unsupported = _bind_params(query, ds)
            if unsupported:
                drow["detail"] = f"unsupported parameters: {unsupported}"
            else:
                state, err = _run_statement(f"SELECT * FROM ({query}) LIMIT 1", params)
                drow["passed"] = state == "SUCCEEDED"
                drow["detail"] = state if drow["passed"] else f"{state}: {err[:250]}"
        except Exception as e:
            drow["detail"] = str(e)[:300]
        results.append(drow)
    ok = sum(1 for r in results if r["check"].startswith("ops_dashboard:dataset") and r["passed"])
    print(f"Ops dashboard datasets: {ok}/{len(datasets)} executed")

In [0]:
from pyspark.sql import Row

result_rows = [Row(check=r["check"], passed=r["passed"], detail=r["detail"]) for r in results]
result_df = spark.createDataFrame(result_rows)
display(result_df)

failures = [r for r in results if not r["passed"]]
if failures:
    summary = "; ".join(f'{r["check"]}: {r["detail"]}' for r in failures)
    raise AssertionError(f"{len(failures)}/{len(results)} check(s) FAILED — {summary}")

print(f"\u2705 All {len(results)} verification checks passed")